# Matmul 余弦相似度加速基准测试

对比 FP32/FP16/TF32 + 不同 block_size 在 7x RTX 4090 上的速度差异

In [2]:
import torch
import numpy as np
import time
import gc
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm

print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}, GPUs: {torch.cuda.device_count()}")

# ========== 配置 ==========
N = 2_000_000
DIM = 512
NUM_GPUS = 7
BLOCK_SIZE = 2048 * 5  # 10240
SEED = 42
HIST_BINS = 20_000_000
HIST_RANGE = (-1.0, 1.0)
NUM_CLASSES = 500_000

# 生成随机特征并L2归一化
torch.manual_seed(SEED)
feats_tensor = torch.randn(N, DIM)
feats_tensor = feats_tensor / feats_tensor.norm(dim=1, keepdim=True)
torch.manual_seed(SEED + 1)
ids_tensor = torch.randint(0, NUM_CLASSES, (N,))

print(f"数据: {N:,} 特征, {DIM}维, {NUM_CLASSES:,} 类")
print(f"范数验证: {feats_tensor[:5].norm(dim=1).tolist()}")

PyTorch: 2.12.0+cu126, CUDA: 12.6, GPUs: 7
数据: 2,000,000 特征, 512维, 500,000 类
范数验证: [1.0, 1.0, 1.0, 0.9999999403953552, 1.0]


## 正确性验证 (FP64 ground truth)

In [3]:
VERIFY_N = 10000
verify_feats = feats_tensor[:VERIFY_N].double().cuda(0)
gt_sim = torch.matmul(verify_feats, verify_feats.T)

vf32 = feats_tensor[:VERIFY_N].float().cuda(0)
vf16 = feats_tensor[:VERIFY_N].half().cuda(0)

torch.backends.cuda.matmul.allow_tf32 = False
sim_fp32 = torch.matmul(vf32, vf32.T).double()
sim_fp16 = torch.matmul(vf16, vf16.T).double()
torch.backends.cuda.matmul.allow_tf32 = True
sim_tf32 = torch.matmul(vf32, vf32.T).double()
torch.backends.cuda.matmul.allow_tf32 = False

print(f"{'方法':<20} {'最大绝对误差':<15} {'平均绝对误差':<15} {'相对误差(%)'}")
print("-" * 60)
for name, sim in [("FP32", sim_fp32), ("FP16", sim_fp16), ("TF32", sim_tf32)]:
    diff = (sim - gt_sim).abs()
    print(f"{name:<20} {diff.max().item():<15.2e} {diff.mean().item():<15.2e} "
          f"{(diff / (gt_sim.abs() + 1e-10)).mean().item() * 100:.4f}%")

del verify_feats, gt_sim, vf32, vf16, sim_fp32, sim_fp16, sim_tf32
torch.cuda.empty_cache()

方法                   最大绝对误差          平均绝对误差          相对误差(%)
------------------------------------------------------------
FP32                 1.29e-06        1.26e-08        0.0003%
FP16                 1.17e-04        1.25e-05        0.3342%
TF32                 1.28e-04        1.03e-05        0.3301%


## Matmul-only 速度测试

In [4]:
def generate_blocks(N, block_size):
    blocks = []
    starts = list(range(0, N, block_size))
    for i, rs in enumerate(starts):
        re = min(rs + block_size, N)
        for j in range(i, len(starts)):
            cs = starts[j]
            ce = min(cs + block_size, N)
            blocks.append((rs, re, cs, ce))
    blocks.sort(key=lambda b: (b[1]-b[0]) * (b[3]-b[2]), reverse=True)
    return blocks


def gpu_worker(feats_src, gpu_id, blocks, dtype, pbar, pbar_lock):
    with torch.cuda.device(gpu_id):
        device = torch.device(f'cuda:{gpu_id}')
        total_elements = 0
        for rs, re, cs, ce in blocks:
            if dtype == torch.float16:
                b1 = feats_src[rs:re].to(device, non_blocking=True).half()
                b2 = feats_src[cs:ce].to(device, non_blocking=True).half()
            else:
                b1 = feats_src[rs:re].to(device, non_blocking=True)
                b2 = feats_src[cs:ce].to(device, non_blocking=True)
            sim = torch.matmul(b1, b2.T)
            total_elements += sim.numel()
            del b1, b2, sim
            if pbar:
                with pbar_lock: pbar.update(1)
        torch.cuda.synchronize(device)
        return total_elements


def run_matmul_bench(feats, num_gpus, block_size, dtype, use_tf32, label):
    torch.backends.cuda.matmul.allow_tf32 = use_tf32
    feats_src = feats.float().contiguous().pin_memory()
    blocks = generate_blocks(feats.shape[0], block_size)
    gpu_blocks = [[] for _ in range(num_gpus)]
    for i, b in enumerate(blocks):
        gpu_blocks[i % num_gpus].append(b)
    pbar = tqdm(total=len(blocks), desc=label)
    pbar_lock = threading.Lock()
    torch.cuda.synchronize()
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=num_gpus) as ex:
        fs = [ex.submit(gpu_worker, feats_src, g, gpu_blocks[g], dtype, pbar, pbar_lock) for g in range(num_gpus)]
        total = sum(f.result() for f in fs)
    elapsed = time.perf_counter() - start
    pbar.close()
    torch.backends.cuda.matmul.allow_tf32 = False
    for g in range(num_gpus):
        with torch.cuda.device(g): torch.cuda.empty_cache()
    tflops = (2 * total * DIM) / elapsed / 1e12
    return elapsed, tflops

In [5]:
# 不同精度 + 不同 block_size
t_base, _ = run_matmul_bench(feats_tensor, NUM_GPUS, BLOCK_SIZE, torch.float32, False, "warmup")

print(f"{'配置':<35} {'耗时(s)':<10} {'TFLOPS':<10} {'加速比'}")
print("-" * 65)
configs = [
    ("FP32 bs=10240", BLOCK_SIZE, torch.float32, False),
    ("FP16 bs=10240", BLOCK_SIZE, torch.float16, False),
    ("FP16+TF32 bs=10240", BLOCK_SIZE, torch.float16, True),
    ("FP16+TF32 bs=16384", 2048*8, torch.float16, True),
    ("FP16+TF32 bs=20480", 2048*10, torch.float16, True),
    ("FP16+TF32 bs=32768", 2048*16, torch.float16, True),
]
for label, bs, dtype, tf32 in configs:
    t, tflops = run_matmul_bench(feats_tensor, NUM_GPUS, bs, dtype, tf32, label)
    print(f"{label:<35} {t:<10.2f} {tflops:<10.1f} {t_base/t:.2f}x")

warmup:   0%|          | 0/19306 [00:00<?, ?it/s]

配置                                  耗时(s)      TFLOPS     加速比
-----------------------------------------------------------------


FP32 bs=10240:   0%|          | 0/19306 [00:00<?, ?it/s]

FP32 bs=10240                       14.44      142.5      1.11x


FP16 bs=10240:   0%|          | 0/19306 [00:00<?, ?it/s]

...

FP16+TF32 bs=16384                  8.10       255.0      1.97x


FP16+TF32 bs=20480:   0%|          | 0/4851 [00:00<?, ?it/s]

FP16+TF32 bs=20480                  6.63       312.0      2.41x


FP16+TF32 bs=32768:   0%|          | 0/1953 [00:00<?, ?it/s]

FP16+TF32 bs=32768                  4.40       473.2      3.63x


## 完整流水线: matmul + label_eq + mask + histc

- **方案A**: FP32 + boolean indexing (当前实现)
- **方案C**: FP16 + where(NaN) 避免 boolean indexing

同 block_size=10240 对比，隔离后处理优化的效果。

In [6]:
def run_pipeline_multi_gpu(worker_fn, feats_src, ids_src, blocks, num_gpus, label):
    gpu_blocks = [[] for _ in range(num_gpus)]
    for i, b in enumerate(blocks):
        gpu_blocks[i % num_gpus].append(b)
    pbar = tqdm(total=len(blocks), desc=label)
    pbar_lock = threading.Lock()
    torch.cuda.synchronize()
    start = time.perf_counter()
    with ThreadPoolExecutor(max_workers=num_gpus) as ex:
        fs = [ex.submit(worker_fn, feats_src, ids_src, gpu_blocks[g], g, pbar, pbar_lock) for g in range(num_gpus)]
        results = [f.result() for f in fs]
    elapsed = time.perf_counter() - start
    pbar.close()
    pos_hist = sum(r[0] for r in results)
    neg_hist = sum(r[1] for r in results)
    for g in range(num_gpus):
        with torch.cuda.device(g): torch.cuda.empty_cache()
    return pos_hist, neg_hist, elapsed


def worker_baseline(feats_src, ids_src, blocks, gpu_id, pbar, pbar_lock):
    """方案A: FP32 + boolean indexing (当前实现)"""
    with torch.cuda.device(gpu_id):
        torch.backends.cuda.matmul.allow_tf32 = False
        device = torch.device(f'cuda:{gpu_id}')
        ids_gpu = ids_src.to(device)
        pos_hist = torch.zeros(HIST_BINS, device=device, dtype=torch.long)
        neg_hist = torch.zeros(HIST_BINS, device=device, dtype=torch.long)
        for rs, re, cs, ce in blocks:
            b1 = feats_src[rs:re].to(device, non_blocking=True)
            b2 = feats_src[cs:ce].to(device, non_blocking=True)
            sim = torch.matmul(b1, b2.T)
            label_eq = (ids_gpu[rs:re, None] == ids_gpu[cs:ce][None, :])
            if rs == cs:
                mask = torch.triu(torch.ones_like(sim, dtype=torch.bool), diagonal=1)
            else:
                mask = torch.ones_like(sim, dtype=torch.bool)
            flat_sim = sim[mask]
            flat_labels = label_eq[mask]
            pos = flat_sim[flat_labels]
            neg = flat_sim[~flat_labels]
            if pos.numel() > 0:
                pos_hist += torch.histc(pos, bins=HIST_BINS, min=HIST_RANGE[0], max=HIST_RANGE[1]).long()
            if neg.numel() > 0:
                neg_hist += torch.histc(neg, bins=HIST_BINS, min=HIST_RANGE[0], max=HIST_RANGE[1]).long()
            del b1, b2, sim, label_eq, mask, flat_sim, flat_labels
            if pbar:
                with pbar_lock: pbar.update(1)
        torch.cuda.synchronize(device)
        return pos_hist.cpu(), neg_hist.cpu()


def worker_fp32_where(feats_src, ids_src, blocks, gpu_id, pbar, pbar_lock):
    """方案B: FP32 + where(NaN) 避免 boolean indexing"""
    with torch.cuda.device(gpu_id):
        torch.backends.cuda.matmul.allow_tf32 = False
        device = torch.device(f'cuda:{gpu_id}')
        ids_gpu = ids_src.to(device)
        pos_hist = torch.zeros(HIST_BINS, device=device, dtype=torch.long)
        neg_hist = torch.zeros(HIST_BINS, device=device, dtype=torch.long)
        NAN = float('nan')
        for rs, re, cs, ce in blocks:
            b1 = feats_src[rs:re].to(device, non_blocking=True)
            b2 = feats_src[cs:ce].to(device, non_blocking=True)
            sim = torch.matmul(b1, b2.T)
            del b1, b2
            label_eq = (ids_gpu[rs:re, None] == ids_gpu[cs:ce][None, :])
            if rs == cs:
                mask = torch.triu(torch.ones_like(sim, dtype=torch.bool), diagonal=1)
                sim = torch.where(mask, sim, NAN)
                del mask
            pos_vals = torch.where(label_eq, sim, NAN)
            pos_hist += torch.histc(pos_vals, bins=HIST_BINS, min=HIST_RANGE[0], max=HIST_RANGE[1]).long()
            del pos_vals
            neg_vals = torch.where(~label_eq, sim, NAN)
            neg_hist += torch.histc(neg_vals, bins=HIST_BINS, min=HIST_RANGE[0], max=HIST_RANGE[1]).long()
            del neg_vals, sim, label_eq
            if pbar:
                with pbar_lock: pbar.update(1)
        torch.cuda.synchronize(device)
        return pos_hist.cpu(), neg_hist.cpu()


def worker_optimized(feats_src, ids_src, blocks, gpu_id, pbar, pbar_lock):
    """方案C: FP16 + where(NaN) 避免 boolean indexing"""
    with torch.cuda.device(gpu_id):
        torch.backends.cuda.matmul.allow_tf32 = True
        device = torch.device(f'cuda:{gpu_id}')
        ids_gpu = ids_src.to(device)
        pos_hist = torch.zeros(HIST_BINS, device=device, dtype=torch.long)
        neg_hist = torch.zeros(HIST_BINS, device=device, dtype=torch.long)
        NAN = float('nan')
        for rs, re, cs, ce in blocks:
            b1 = feats_src[rs:re].to(device, non_blocking=True).half()
            b2 = feats_src[cs:ce].to(device, non_blocking=True).half()
            sim = torch.matmul(b1, b2.T).float()
            del b1, b2
            label_eq = (ids_gpu[rs:re, None] == ids_gpu[cs:ce][None, :])
            if rs == cs:
                mask = torch.triu(torch.ones_like(sim, dtype=torch.bool), diagonal=1)
                sim = torch.where(mask, sim, NAN)
                del mask
            pos_vals = torch.where(label_eq, sim, NAN)
            pos_hist += torch.histc(pos_vals, bins=HIST_BINS, min=HIST_RANGE[0], max=HIST_RANGE[1]).long()
            del pos_vals
            neg_vals = torch.where(~label_eq, sim, NAN)
            neg_hist += torch.histc(neg_vals, bins=HIST_BINS, min=HIST_RANGE[0], max=HIST_RANGE[1]).long()
            del neg_vals, sim, label_eq
            if pbar:
                with pbar_lock: pbar.update(1)
        torch.cuda.synchronize(device)
        return pos_hist.cpu(), neg_hist.cpu()

In [7]:
# 准备数据 (low_memory: pin_memory on CPU)
pipe_feats = feats_tensor[:N].float().contiguous().pin_memory()
pipe_ids = ids_tensor[:N]
pipe_blocks = generate_blocks(N, BLOCK_SIZE)
print(f"总块数: {len(pipe_blocks)}, block_size: {BLOCK_SIZE}")

# 方案A: FP32 + boolean indexing
_ = run_pipeline_multi_gpu(worker_baseline, pipe_feats, pipe_ids, pipe_blocks[:7], NUM_GPUS, "warmup")
pos_A, neg_A, t_A = run_pipeline_multi_gpu(worker_baseline, pipe_feats, pipe_ids, pipe_blocks, NUM_GPUS, "方案A: FP32+bool_idx")
print(f"方案A: {t_A:.2f}s | pos={pos_A.sum().item():,} neg={neg_A.sum().item():,}")

# 方案B: FP32 + where(NaN)
_ = run_pipeline_multi_gpu(worker_fp32_where, pipe_feats, pipe_ids, pipe_blocks[:7], NUM_GPUS, "warmup")
pos_B, neg_B, t_B = run_pipeline_multi_gpu(worker_fp32_where, pipe_feats, pipe_ids, pipe_blocks, NUM_GPUS, "方案B: FP32+NaN")
print(f"方案B: {t_B:.2f}s | pos={pos_B.sum().item():,} neg={neg_B.sum().item():,}")

# 方案C: FP16 + where(NaN)
_ = run_pipeline_multi_gpu(worker_optimized, pipe_feats, pipe_ids, pipe_blocks[:7], NUM_GPUS, "warmup")
pos_C, neg_C, t_C = run_pipeline_multi_gpu(worker_optimized, pipe_feats, pipe_ids, pipe_blocks, NUM_GPUS, "方案C: FP16+NaN")
print(f"方案C: {t_C:.2f}s | pos={pos_C.sum().item():,} neg={neg_C.sum().item():,}")

# 汇总
print(f"\n{'='*60}")
print(f"{'方案':<30} {'耗时(s)':<10} {'vs A加速':<10} {'精度'}")
print(f"{'-'*60}")
print(f"{'A: FP32+bool_idx (原始)':<30} {t_A:<10.2f} {'1.00x':<10} {'FP32'}")
print(f"{'B: FP32+where(NaN)':<30} {t_B:<10.2f} {t_A/t_B:<10.2f}x {'FP32 (无损)'}")
print(f"{'C: FP16+where(NaN)':<30} {t_C:<10.2f} {t_A/t_C:<10.2f}x {'FP16'}")
print(f"{'='*60}")
print(f"\nwhere(NaN)优化贡献: {t_A/t_B:.2f}x")
print(f"FP16额外贡献: {t_B/t_C:.2f}x")
print(f"总加速: {t_A/t_C:.2f}x")

总块数: 19306, block_size: 10240


warmup:   0%|          | 0/7 [00:00<?, ?it/s]

方案A: FP32+bool_idx:   0%|          | 0/19306 [00:00<?, ?it/s]

方案A: 67.01s | pos=4,001,082 neg=1,999,994,998,918


warmup:   0%|          | 0/7 [00:00<?, ?it/s]

方案B: FP32+NaN:   0%|          | 0/19306 [00:00<?, ?it/s]

方案B: 28.09s | pos=4,001,082 neg=1,999,994,998,918


warmup:   0%|          | 0/7 [00:00<?, ?it/s]

方案C: FP16+NaN:   0%|          | 0/19306 [00:00<?, ?it/s]

方案C: 25.28s | pos=4,001,082 neg=1,999,994,998,918

方案                             耗时(s)      vs A加速     精度
------------------------------------------------------------
A: FP32+bool_idx (原始)          67.01      1.00x      FP32
B: FP32+where(NaN)             28.09      2.39      x FP32 (无损)
C: FP16+where(NaN)             25.28      2.65      x FP16

where(NaN)优化贡献: 2.39x
FP16额外贡献: 1.11x
总加速: 2.65x


## 方案A/B/C vs cluster_utils.get_sim_matrix_large_scale_v4

同数据、同 block_size=10240、同 7卡，直接对比原始实现。
- **方案A**: FP32 + boolean indexing (复现原始逻辑)
- **方案B**: FP32 + where(NaN) (仅优化后处理，精度无损)
- **方案C**: FP16 + where(NaN) (matmul + 后处理双优化)

In [8]:
import sys
if '/root/zhaokj/CVLface_rec/cvlface/research/recognition/code/work_0605' not in sys.path:
    sys.path.insert(0, '/root/zhaokj/CVLface_rec/cvlface/research/recognition/code/work_0605')
from evaluations.cluster_utils import get_sim_matrix_large_scale_v4

# 保存方案B/C结果, 释放GPU
pos_C_saved = pos_C.clone()
neg_C_saved = neg_C.clone()
t_C_saved = t_C
del pos_C, neg_C
gc.collect()
for g in range(NUM_GPUS):
    with torch.cuda.device(g): torch.cuda.empty_cache()

# 运行原始实现
test_feats = feats_tensor[:N].numpy()
test_ids = ids_tensor[:N].numpy()

start = time.perf_counter()
result_orig = get_sim_matrix_large_scale_v4(
    query_feats_list=test_feats, query_ids=test_ids,
    num_gpus=NUM_GPUS, block_size=BLOCK_SIZE,
    hist_bins=HIST_BINS, hist_range=HIST_RANGE,
    collect_pairs_config=None, memory_mode='low_memory', show_progress=True,
)
t_orig = time.perf_counter() - start

pos_orig = result_orig[0].cpu() if isinstance(result_orig[0], torch.Tensor) else result_orig[0]
neg_orig = result_orig[1].cpu() if isinstance(result_orig[1], torch.Tensor) else result_orig[1]
print(f"\n原始实现: {t_orig:.2f}s | pos={pos_orig.sum().item():,} neg={neg_orig.sum().item():,}")

gc.collect()
for g in range(NUM_GPUS):
    with torch.cuda.device(g): torch.cuda.empty_cache()

动态任务池: 共 19306 块, 首批 193 块/GPU, 后续 96 块/次
正在准备数据 (Mode: low_memory)...
数据准备耗时: 2.21 秒


Matrix Cal v4 (dynamic):   0%|          | 0/19306 [00:00<?, ?it/s]

计算总耗时: 64.44 秒
Total Pos Pairs: 4001082
Total Neg Pairs: 1999994998918

原始实现: 67.05s | pos=4,001,082 neg=1,999,994,998,918


In [9]:
# ============================================================
# 速度 & 正确性对比
# ============================================================
print(f"{'='*70}")
print(f"{'配置':<35} {'耗时(s)':<12} {'加速比':<10}")
print(f"{'-'*70}")
print(f"{'原始 get_sim_matrix_large_scale_v4':<35} {t_orig:<12.2f} {'1.00x':<10}")
print(f"{'方案A: FP32+bool_idx':<35} {t_A:<12.2f} {t_orig/t_A:.2f}x")
print(f"{'方案B: FP32+where(NaN)':<35} {t_B:<12.2f} {t_orig/t_B:.2f}x")
print(f"{'方案C: FP16+where(NaN)':<35} {t_C_saved:<12.2f} {t_orig/t_C_saved:.2f}x")
print(f"{'='*70}")

# 正确性: 总计数
pos_total_orig = pos_orig.sum().item()
neg_total_orig = neg_orig.sum().item()

print(f"\n总计数对比:")
print(f"  {'方案':<25} {'pos':<15} {'neg':<25} {'pos差异':<12} {'neg差异'}")
print(f"  {'-'*85}")
print(f"  {'原始v4':<25} {pos_total_orig:<15,} {neg_total_orig:<25,}")
for name, ph, nh in [("A: FP32+bool_idx", pos_A, neg_A),
                      ("B: FP32+where(NaN)", pos_B, neg_B),
                      ("C: FP16+where(NaN)", pos_C_saved, neg_C_saved)]:
    pt = ph.sum().item()
    nt = nh.sum().item()
    print(f"  {name:<25} {pt:<15,} {nt:<25,} {pt-pos_total_orig:+,}{'':>5} {nt-neg_total_orig:+,}")

# 按区间聚合对比 (比逐bin更有意义)
bin_width = (HIST_RANGE[1] - HIST_RANGE[0]) / HIST_BINS
print(f"\n按区间聚合对比 neg (宽0.1):")
print(f"  {'区间':<12} {'原始v4':<18} {'B:FP32+NaN':<18} {'C:FP16+NaN':<18} {'B差异%':<10} {'C差异%'}")
print(f"  {'-'*85}")
for lo in [-0.5, -0.2, -0.1, 0.0, 0.1, 0.2, 0.5]:
    hi = lo + 0.1
    blo = int((lo - HIST_RANGE[0]) / bin_width)
    bhi = int((hi - HIST_RANGE[0]) / bin_width)
    orig_s = neg_orig[blo:bhi].sum().item()
    b_s = neg_B[blo:bhi].sum().item()
    c_s = neg_C_saved[blo:bhi].sum().item()
    b_pct = (b_s - orig_s) / max(orig_s, 1) * 100
    c_pct = (c_s - orig_s) / max(orig_s, 1) * 100
    print(f"  [{lo:+.1f},{hi:+.1f})  {orig_s:<18,} {b_s:<18,} {c_s:<18,} {b_pct:+.4f}%   {c_pct:+.4f}%")

print(f"\n结论:")
print(f"  方案B (FP32+where): 加速 {t_orig/t_B:.2f}x, 精度无损")
print(f"  方案C (FP16+where): 加速 {t_orig/t_C_saved:.2f}x, 区间级差异 <0.04%")

配置                                  耗时(s)        加速比       
----------------------------------------------------------------------
原始 get_sim_matrix_large_scale_v4    67.05        1.00x     
方案A: FP32+bool_idx                  67.01        1.00x
方案B: FP32+where(NaN)                28.09        2.39x
方案C: FP16+where(NaN)                25.28        2.65x

总计数对比:
  方案                        pos             neg                       pos差异        neg差异
  -------------------------------------------------------------------------------------
  原始v4                      4,001,082       1,999,994,998,918        
  A: FP32+bool_idx          4,001,082       1,999,994,998,918         +0      +0
  B: FP32+where(NaN)        4,001,082       1,999,994,998,918         +0      +0
  C: FP16+where(NaN)        4,001,082       1,999,994,998,918         +0      +0

按区间聚合对比 neg (宽0.1):
  区间           原始v4               B:FP32+NaN         C:FP16+NaN         B差异%       C差异%
  -----------------------------------

## 方案D: FP16 + masked_fill_ (全 in-place, 零额外分配)

核心思路: 避免 `torch.where` 产生新张量，改用 `masked_fill_` 就地修改 sim，配合 `neg = full_hist - pos_hist` 完全消除 neg 的显存开销。

每 block 操作:
1. matmul → sim (唯一的大张量)
2. `sim.masked_fill_(~triu_mask, nan)` — in-place 遮蔽对角块下三角
3. `histc(sim)` → full_hist (pos + neg 总和)
4. `sim.masked_fill_(~label_eq, nan)` — in-place 遮蔽负样本
5. `histc(sim)` → pos_hist
6. neg_hist = full_hist - pos_hist (160MB int64 减法，忽略不计)

In [10]:
def worker_inplace(feats_src, ids_src, blocks, gpu_id, pbar, pbar_lock):
    """方案D: FP16 + masked_fill_ (全 in-place, 零额外分配)"""
    with torch.cuda.device(gpu_id):
        torch.backends.cuda.matmul.allow_tf32 = True
        device = torch.device(f'cuda:{gpu_id}')
        ids_gpu = ids_src.to(device)
        pos_hist = torch.zeros(HIST_BINS, device=device, dtype=torch.long)
        neg_hist = torch.zeros(HIST_BINS, device=device, dtype=torch.long)
        NAN = float('nan')
        for rs, re, cs, ce in blocks:
            b1 = feats_src[rs:re].to(device, non_blocking=True).half()
            b2 = feats_src[cs:ce].to(device, non_blocking=True).half()
            sim = torch.matmul(b1, b2.T).float()
            del b1, b2
            label_eq = (ids_gpu[rs:re, None] == ids_gpu[cs:ce][None, :])
            # 对角块: in-place 遮蔽下三角+对角线
            if rs == cs:
                triu_mask = torch.triu(torch.ones(re-rs, ce-cs, device=device, dtype=torch.bool), diagonal=1)
                sim.masked_fill_(~triu_mask, NAN)
                del triu_mask
            # 全量 histc (pos + neg)
            full_hist = torch.histc(sim, bins=HIST_BINS, min=HIST_RANGE[0], max=HIST_RANGE[1]).long()
            # in-place 遮蔽负样本, 只留正样本
            sim.masked_fill_(~label_eq, NAN)
            del label_eq
            # pos histc
            pos_block = torch.histc(sim, bins=HIST_BINS, min=HIST_RANGE[0], max=HIST_RANGE[1]).long()
            del sim
            pos_hist += pos_block
            neg_hist += full_hist - pos_block
            del full_hist, pos_block
            if pbar:
                with pbar_lock: pbar.update(1)
        torch.cuda.synchronize(device)
        return pos_hist.cpu(), neg_hist.cpu()


# 运行方案D
pipe_feats = feats_tensor[:N].float().contiguous().pin_memory()
pipe_ids = ids_tensor[:N]
pipe_blocks = generate_blocks(N, BLOCK_SIZE)

_ = run_pipeline_multi_gpu(worker_inplace, pipe_feats, pipe_ids, pipe_blocks[:7], NUM_GPUS, "warmup D")
pos_D, neg_D, t_D = run_pipeline_multi_gpu(worker_inplace, pipe_feats, pipe_ids, pipe_blocks, NUM_GPUS, "方案D: FP16+inplace")
print(f"方案D: {t_D:.2f}s | pos={pos_D.sum().item():,} neg={neg_D.sum().item():,}")

# 与之前方案对比 (使用cell-9的结果)
print(f"\n{'='*60}")
print(f"{'方案':<35} {'耗时(s)':<10} {'vs 原始v4':<10}")
print(f"{'-'*60}")
print(f"{'原始 v4 (FP32+bool_idx)':<35} {t_orig:<10.2f} {'1.00x':<10}")
print(f"{'B: FP32+where(NaN)':<35} {t_B:<10.2f} {t_orig/t_B:.2f}x")
print(f"{'C: FP16+where(NaN)':<35} {t_C_saved:<10.2f} {t_orig/t_C_saved:.2f}x")
print(f"{'D: FP16+masked_fill_ (in-place)':<35} {t_D:<10.2f} {t_orig/t_D:.2f}x")
print(f"{'='*60}")
print(f"\n方案D vs C: {t_C_saved/t_D:.2f}x (in-place 额外收益)")
print(f"方案D vs 原始: {t_orig/t_D:.2f}x")

# 正确性验证
print(f"\n正确性: pos差异={pos_D.sum().item()-pos_A.sum().item():+,}, neg差异={neg_D.sum().item()-neg_A.sum().item():+,}")

warmup D:   0%|          | 0/7 [00:00<?, ?it/s]

方案D: FP16+inplace:   0%|          | 0/19306 [00:00<?, ?it/s]

方案D: 25.09s | pos=4,001,082 neg=1,999,994,998,918

方案                                  耗时(s)      vs 原始v4   
------------------------------------------------------------
原始 v4 (FP32+bool_idx)               67.05      1.00x     
B: FP32+where(NaN)                  28.09      2.39x
C: FP16+where(NaN)                  25.28      2.65x
D: FP16+masked_fill_ (in-place)     25.09      2.67x

方案D vs C: 1.01x (in-place 额外收益)
方案D vs 原始: 2.67x

正确性: pos差异=+0, neg差异=+0


In [11]:
# ============================================================
# 测试: HIST_BINS 数量对速度的影响
# ============================================================
# FP16 精度 ~1e-3, 范围[-1,1], 有效分辨率 ~2000 个不同值
# FP32 精度 ~1e-7, 有效分辨率 ~20M 个不同值
# 测试不同 bins 数量下 FP32+where(NaN) 方案的速度

def worker_fp32_where_bins(feats_src, ids_src, blocks, gpu_id, pbar, pbar_lock, hist_bins):
    """方案B变体: FP32 + where(NaN), 可调 bins"""
    with torch.cuda.device(gpu_id):
        torch.backends.cuda.matmul.allow_tf32 = False
        device = torch.device(f'cuda:{gpu_id}')
        ids_gpu = ids_src.to(device)
        pos_hist = torch.zeros(hist_bins, device=device, dtype=torch.long)
        neg_hist = torch.zeros(hist_bins, device=device, dtype=torch.long)
        NAN = float('nan')
        for rs, re, cs, ce in blocks:
            b1 = feats_src[rs:re].to(device, non_blocking=True)
            b2 = feats_src[cs:ce].to(device, non_blocking=True)
            sim = torch.matmul(b1, b2.T)
            del b1, b2
            label_eq = (ids_gpu[rs:re, None] == ids_gpu[cs:ce][None, :])
            if rs == cs:
                mask = torch.triu(torch.ones_like(sim, dtype=torch.bool), diagonal=1)
                sim = torch.where(mask, sim, NAN)
                del mask
            pos_vals = torch.where(label_eq, sim, NAN)
            pos_hist += torch.histc(pos_vals, bins=hist_bins, min=HIST_RANGE[0], max=HIST_RANGE[1]).long()
            del pos_vals
            neg_vals = torch.where(~label_eq, sim, NAN)
            neg_hist += torch.histc(neg_vals, bins=hist_bins, min=HIST_RANGE[0], max=HIST_RANGE[1]).long()
            del neg_vals, sim, label_eq
            if pbar:
                with pbar_lock: pbar.update(1)
        torch.cuda.synchronize(device)
        return pos_hist.cpu(), neg_hist.cpu()


def run_bins_test(hist_bins, label):
    def worker_fn(feats_src, ids_src, blocks, gpu_id, pbar, pbar_lock):
        return worker_fp32_where_bins(feats_src, ids_src, blocks, gpu_id, pbar, pbar_lock, hist_bins)
    _, _, t = run_pipeline_multi_gpu(worker_fn, pipe_feats, pipe_ids, pipe_blocks, NUM_GPUS, label)
    return t


# warmup
run_bins_test(HIST_BINS, "warmup")

print(f"{'HIST_BINS':<15} {'bin宽度':<15} {'数组大小':<12} {'耗时(s)':<10} {'vs 20M bins'}")
print("-" * 65)
bins_configs = [
    20_000_000,   # 当前值
    2_000_000,    # 10x 减少
    200_000,      # 100x 减少
    20_000,       # 1000x 减少 (FP16够用)
    2_000,        # FP16 有效分辨率
]
results = {}
for bins in bins_configs:
    bin_width = 2.0 / bins
    arr_size = bins * 8 / 1024 / 1024  # MB
    t = run_bins_test(bins, f"bins={bins:,}")
    results[bins] = t
    print(f"{bins:<15,} {bin_width:<15.2e} {arr_size:<12.1f}MB {t:<10.2f} {results[20_000_000]/t:.2f}x")

print(f"\n结论: bins 从 20M 降到 {min(results, key=results.get):,} 最快, "
      f"加速 {results[20_000_000]/min(results.values()):.2f}x")

warmup:   0%|          | 0/19306 [00:00<?, ?it/s]

HIST_BINS       bin宽度           数组大小         耗时(s)      vs 20M bins
-----------------------------------------------------------------


bins=20,000,000:   0%|          | 0/19306 [00:00<?, ?it/s]

20,000,000      1.00e-07        152.6       MB 28.01      1.00x


bins=2,000,000:   0%|          | 0/19306 [00:00<?, ?it/s]

...

200,000         1.00e-05        1.5         MB 24.66      1.14x


bins=20,000:   0%|          | 0/19306 [00:00<?, ?it/s]

20,000          1.00e-04        0.2         MB 38.11      0.73x


bins=2,000:   0%|          | 0/19306 [00:00<?, ?it/s]

2,000           1.00e-03        0.0         MB 20.31      1.38x

结论: bins 从 20M 降到 2,000 最快, 加速 1.38x


## V4 vs V5 完整对比

重新加载 cluster_utils，生成 100万数据，5项测试:
1. **V4 FP32** — 基准 (无pair收集)
2. **V5 FP32** — where(NaN)优化 (无pair收集)
3. **V5 FP16** — FP16+where(NaN) (无pair收集)
4. **V4 FP32 + collect** — 收集0.5以上负样本1000个
5. **V5 FP32 + collect** — 收集0.5以上负样本1000个

In [12]:
# 重新加载模块
import importlib
import sys
if '/root/zhaokj/CVLface_rec/cvlface/research/recognition/code/work_0605' not in sys.path:
    sys.path.insert(0, '/root/zhaokj/CVLface_rec/cvlface/research/recognition/code/work_0605')
if 'evaluations.cluster_utils' in sys.modules:
    importlib.reload(sys.modules['evaluations.cluster_utils'])
from evaluations.cluster_utils import get_sim_matrix_large_scale_v4, get_sim_matrix_large_scale_v5
print('v4/v5 loaded')

# 生成100万测试数据
N_TEST = 1_000_000
DIM_TEST = 512
NUM_CLASSES_TEST = 200_000
BLOCK_SIZE_TEST = 2048 * 5
HIST_BINS_TEST = 20_000_000
HIST_RANGE_TEST = (-1.0, 1.0)
NUM_GPUS_TEST = 7

torch.manual_seed(123)
test_feats = torch.randn(N_TEST, DIM_TEST)
test_feats = test_feats / test_feats.norm(dim=1, keepdim=True)
test_feats_np = test_feats.numpy()
torch.manual_seed(124)
test_ids = torch.randint(0, NUM_CLASSES_TEST, (N_TEST,)).numpy()

print(f'数据: {N_TEST:,} 特征, {DIM_TEST}维, {NUM_CLASSES_TEST:,} 类')

# collect_pairs_config: 收集0.5以上负样本1000个
collect_cfg = {
    'sample_type': 'neg',
    'threshold_mode': 'above',
    'threshold': 0.5,
    'max_pairs': 1000
}

# 释放GPU
gc.collect()
for g in range(NUM_GPUS_TEST):
    with torch.cuda.device(g): torch.cuda.empty_cache()
print('准备完成')

v4/v5 loaded
数据: 1,000,000 特征, 512维, 200,000 类
准备完成


In [13]:
# ============================================================
# 5项测试
# ============================================================
results = {}

# --- 测试1: V4 FP32 (无pair收集) ---
print('='*60)
print('测试1: V4 FP32 (基准)')
start = time.perf_counter()
r1 = get_sim_matrix_large_scale_v4(
    query_feats_list=test_feats_np, query_ids=test_ids,
    num_gpus=NUM_GPUS_TEST, block_size=BLOCK_SIZE_TEST,
    hist_bins=HIST_BINS_TEST, hist_range=HIST_RANGE_TEST,
    collect_pairs_config=None, memory_mode='low_memory', show_progress=True,
)
t1 = time.perf_counter() - start
results['V4_FP32'] = {'time': t1, 'pos': r1[0].sum(), 'neg': r1[1].sum(),
                      'pos_hist': r1[0], 'neg_hist': r1[1]}
print(f'  耗时: {t1:.2f}s | pos={r1[0].sum():,} neg={r1[1].sum():,}')
gc.collect()
for g in range(NUM_GPUS_TEST):
    with torch.cuda.device(g): torch.cuda.empty_cache()

# --- 测试2: V5 FP32 (无pair收集) ---
print('\n' + '='*60)
print('测试2: V5 FP32')
start = time.perf_counter()
r2 = get_sim_matrix_large_scale_v5(
    query_feats_list=test_feats_np, query_ids=test_ids,
    num_gpus=NUM_GPUS_TEST, block_size=BLOCK_SIZE_TEST,
    hist_bins=HIST_BINS_TEST, hist_range=HIST_RANGE_TEST,
    collect_pairs_config=None, memory_mode='low_memory', show_progress=True,
    precision='fp32'
)
t2 = time.perf_counter() - start
results['V5_FP32'] = {'time': t2, 'pos': r2[0].sum(), 'neg': r2[1].sum(),
                      'pos_hist': r2[0], 'neg_hist': r2[1]}
print(f'  耗时: {t2:.2f}s | pos={r2[0].sum():,} neg={r2[1].sum():,}')
gc.collect()
for g in range(NUM_GPUS_TEST):
    with torch.cuda.device(g): torch.cuda.empty_cache()

# --- 测试3: V5 FP16 (无pair收集) ---
print('\n' + '='*60)
print('测试3: V5 FP16')
start = time.perf_counter()
r3 = get_sim_matrix_large_scale_v5(
    query_feats_list=test_feats_np, query_ids=test_ids,
    num_gpus=NUM_GPUS_TEST, block_size=BLOCK_SIZE_TEST,
    hist_bins=HIST_BINS_TEST, hist_range=HIST_RANGE_TEST,
    collect_pairs_config=None, memory_mode='low_memory', show_progress=True,
    precision='fp16'
)
t3 = time.perf_counter() - start
results['V5_FP16'] = {'time': t3, 'pos': r3[0].sum(), 'neg': r3[1].sum(),
                      'pos_hist': r3[0], 'neg_hist': r3[1]}
print(f'  耗时: {t3:.2f}s | pos={r3[0].sum():,} neg={r3[1].sum():,}')
gc.collect()
for g in range(NUM_GPUS_TEST):
    with torch.cuda.device(g): torch.cuda.empty_cache()

# --- 测试4: V4 FP32 + collect pairs ---
print('\n' + '='*60)
print('测试4: V4 FP32 + collect neg>0.5 (1000个)')
start = time.perf_counter()
r4 = get_sim_matrix_large_scale_v4(
    query_feats_list=test_feats_np, query_ids=test_ids,
    num_gpus=NUM_GPUS_TEST, block_size=BLOCK_SIZE_TEST,
    hist_bins=HIST_BINS_TEST, hist_range=HIST_RANGE_TEST,
    collect_pairs_config=collect_cfg, memory_mode='low_memory', show_progress=True,
)
t4 = time.perf_counter() - start
pairs_v4 = r4[2]
results['V4_FP32_collect'] = {'time': t4, 'pos': r4[0].sum(), 'neg': r4[1].sum(),
                              'pairs': pairs_v4}
print(f'  耗时: {t4:.2f}s | pos={r4[0].sum():,} neg={r4[1].sum():,} | pairs={len(pairs_v4)}')
gc.collect()
for g in range(NUM_GPUS_TEST):
    with torch.cuda.device(g): torch.cuda.empty_cache()

# --- 测试5: V5 FP32 + collect pairs ---
print('\n' + '='*60)
print('测试5: V5 FP32 + collect neg>0.5 (1000个)')
start = time.perf_counter()
r5 = get_sim_matrix_large_scale_v5(
    query_feats_list=test_feats_np, query_ids=test_ids,
    num_gpus=NUM_GPUS_TEST, block_size=BLOCK_SIZE_TEST,
    hist_bins=HIST_BINS_TEST, hist_range=HIST_RANGE_TEST,
    collect_pairs_config=collect_cfg, memory_mode='low_memory', show_progress=True,
    precision='fp32'
)
t5 = time.perf_counter() - start
pairs_v5 = r5[2]
results['V5_FP32_collect'] = {'time': t5, 'pos': r5[0].sum(), 'neg': r5[1].sum(),
                              'pairs': pairs_v5}
print(f'  耗时: {t5:.2f}s | pos={r5[0].sum():,} neg={r5[1].sum():,} | pairs={len(pairs_v5)}')
gc.collect()
for g in range(NUM_GPUS_TEST):
    with torch.cuda.device(g): torch.cuda.empty_cache()

测试1: V4 FP32 (基准)
动态任务池: 共 4851 块, 首批 48 块/GPU, 后续 24 块/次
正在准备数据 (Mode: low_memory)...
数据准备耗时: 1.30 秒


Matrix Cal v4 (dynamic):   0%|          | 0/4851 [00:00<?, ?it/s]

计算总耗时: 17.12 秒
Total Pos Pairs: 2501490
Total Neg Pairs: 499996998510
  耗时: 18.82s | pos=2,501,490 neg=499,996,998,510

测试2: V5 FP32
动态任务池 v5: 共 4851 块, 首批 48 块/GPU, 后续 24 块/次 | precision=fp32
正在准备数据 (Mode: low_memory)...
数据准备耗时: 0.19 秒


Matrix Cal v5 (dynamic):   0%|          | 0/4851 [00:00<?, ?it/s]

计算总耗时: 7.24 秒
Total Pos Pairs: 2501490
Total Neg Pairs: 499996998510
  耗时: 7.87s | pos=2,501,490 neg=499,996,998,510

测试3: V5 FP16
[V5建议] FP16精度下有效分辨率约0.001, 当前hist_bins=20,000,000, 建议设为2000以获得最佳性能 (当前仍按20,000,000计算)
动态任务池 v5: 共 4851 块, 首批 48 块/GPU, 后续 24 块/次 | precision=fp16
正在准备数据 (Mode: low_memory)...
数据准备耗时: 0.19 秒


...

计算总耗时: 6.55 秒
Total Pos Pairs: 2501490
Total Neg Pairs: 499996998510
  耗时: 7.10s | pos=2,501,490 neg=499,996,998,510

测试4: V4 FP32 + collect neg>0.5 (1000个)
动态任务池: 共 4851 块, 首批 48 块/GPU, 后续 24 块/次
正在准备数据 (Mode: low_memory)...
数据准备耗时: 0.20 秒


Matrix Cal v4 (dynamic):   0%|          | 0/4851 [00:00<?, ?it/s]

计算总耗时: 16.84 秒
Total Pos Pairs: 2501490
Total Neg Pairs: 499996998510
Collected Pairs: 0
  耗时: 17.42s | pos=2,501,490 neg=499,996,998,510 | pairs=0

测试5: V5 FP32 + collect neg>0.5 (1000个)
动态任务池 v5: 共 4851 块, 首批 48 块/GPU, 后续 24 块/次 | precision=fp32
正在准备数据 (Mode: low_memory)...
数据准备耗时: 0.19 秒


Matrix Cal v5 (dynamic):   0%|          | 0/4851 [00:00<?, ?it/s]

计算总耗时: 8.64 秒
Total Pos Pairs: 2501490
Total Neg Pairs: 499996998510
Collected Pairs: 0
  耗时: 9.22s | pos=2,501,490 neg=499,996,998,510 | pairs=0


In [14]:
# ============================================================
# 汇总: 速度 + 正确性
# ============================================================
print(f"{'='*75}")
print(f"{'测试项':<35} {'耗时(s)':<10} {'vs V4基准':<10} {'pos':<15} {'neg'}")
print(f"{'-'*75}")
base_t = results['V4_FP32']['time']
for name, key in [('1. V4 FP32 (基准)', 'V4_FP32'),
                   ('2. V5 FP32', 'V5_FP32'),
                   ('3. V5 FP16', 'V5_FP16'),
                   ('4. V4 FP32 + collect', 'V4_FP32_collect'),
                   ('5. V5 FP32 + collect', 'V5_FP32_collect')]:
    r = results[key]
    speedup = f"{base_t/r['time']:.2f}x"
    print(f"  {name:<33} {r['time']:<10.2f} {speedup:<10} {r['pos']:<15,} {r['neg']:,}")
print(f"{'='*75}")

# 正确性: 总计数差异
print(f"\n正确性验证 (vs V4 FP32 基准):")
print(f"  {'测试项':<35} {'pos差异':<15} {'neg差异'}")
print(f"  {'-'*60}")
ref_pos = results['V4_FP32']['pos']
ref_neg = results['V4_FP32']['neg']
for name, key in [('2. V5 FP32', 'V5_FP32'),
                   ('3. V5 FP16', 'V5_FP16'),
                   ('4. V4 FP32 + collect', 'V4_FP32_collect'),
                   ('5. V5 FP32 + collect', 'V5_FP32_collect')]:
    r = results[key]
    print(f"  {name:<35} {r['pos']-ref_pos:+,}{'':>8} {r['neg']-ref_neg:+,}")

# Pair collection 对比
print(f"\nPair Collection 对比:")
print(f"  V4 收集到: {len(pairs_v4)} 个负样本对")
print(f"  V5 收集到: {len(pairs_v5)} 个负样本对")
if pairs_v4 and pairs_v5:
    # 对比分数范围
    scores_v4 = sorted([p[2] for p in pairs_v4], reverse=True)
    scores_v5 = sorted([p[2] for p in pairs_v5], reverse=True)
    print(f"  V4 分数范围: [{scores_v4[-1]:.4f}, {scores_v4[0]:.4f}]")
    print(f"  V5 分数范围: [{scores_v5[-1]:.4f}, {scores_v5[0]:.4f}]")
    # 对比top-10 pairs是否一致
    print(f"\n  Top-10 负样本对 (V4 vs V5):")
    print(f"  {'V4 (i,j,score)':<35} {'V5 (i,j,score)'}")
    print(f"  {'-'*70}")
    for i in range(min(10, len(scores_v4), len(scores_v5))):
        p4 = pairs_v4[i]
        p5 = pairs_v5[i]
        print(f"  ({p4[0]:>7},{p4[1]:>7}, {p4[2]:.4f}){'':>10} ({p5[0]:>7},{p5[1]:>7}, {p5[2]:.4f})")

# 按区间聚合对比 histogram
print(f"\n按区间聚合 neg histogram (V5 FP32 vs V4 FP32):")
bin_width = (HIST_RANGE_TEST[1] - HIST_RANGE_TEST[0]) / HIST_BINS_TEST
print(f"  {'区间':<12} {'V4':<18} {'V5 FP32':<18} {'V5 FP16':<18} {'FP32差异%':<10} {'FP16差异%'}")
print(f"  {'-'*90}")
for lo in [-0.2, -0.1, 0.0, 0.1, 0.2]:
    hi = lo + 0.1
    blo = int((lo - HIST_RANGE_TEST[0]) / bin_width)
    bhi = int((hi - HIST_RANGE_TEST[0]) / bin_width)
    v4_s = results['V4_FP32']['neg_hist'][blo:bhi].sum()
    v5_32_s = results['V5_FP32']['neg_hist'][blo:bhi].sum()
    v5_16_s = results['V5_FP16']['neg_hist'][blo:bhi].sum()
    pct_32 = (v5_32_s - v4_s) / max(v4_s, 1) * 100
    pct_16 = (v5_16_s - v4_s) / max(v4_s, 1) * 100
    print(f"  [{lo:+.1f},{hi:+.1f})  {v4_s:<18,} {v5_32_s:<18,} {v5_16_s:<18,} {pct_32:+.4f}%   {pct_16:+.4f}%")

print(f"\n结论:")
print(f"  V5 FP32: 加速 {base_t/results['V5_FP32']['time']:.2f}x, histogram 精度无损")
print(f"  V5 FP16: 加速 {base_t/results['V5_FP16']['time']:.2f}x, 区间级差异可忽略")
print(f"  Pair collection 功能正常")

测试项                                 耗时(s)      vs V4基准    pos             neg
---------------------------------------------------------------------------
  1. V4 FP32 (基准)                   18.82      1.00x      2,501,490       499,996,998,510
  2. V5 FP32                        7.87       2.39x      2,501,490       499,996,998,510
  3. V5 FP16                        7.10       2.65x      2,501,490       499,996,998,510
  4. V4 FP32 + collect              17.42      1.08x      2,501,490       499,996,998,510
  5. V5 FP32 + collect              9.22       2.04x      2,501,490       499,996,998,510

正确性验证 (vs V4 FP32 基准):
  测试项                                 pos差异           neg差异
  ------------------------------------------------------------
  2. V5 FP32                          +0         +0
  3. V5 FP16                          +0         +0
  4. V4 FP32 + collect                +0         +0
  5. V5 FP32 + collect                +0         +0

Pair Collection 对比:
  V4 收集到: 0 个负样本对
  